# PROC FORECAST による自動車保険月次請求件数予測

## エグゼクティブサマリー

損害準備金設定チームは、着実なポートフォリオ成長と顕著な冬季悪天候による増加を示す、自動車保険の月次請求件数について12ヶ月先までの見通しを必要としている。このノートブックは5年分（60ヶ月、2020年1月～2024年12月、約1,460件～2,780件の範囲）の合成月次請求件数を生成し、**PROC FORECAST** を用いてステップワイズ自己回帰ベースラインモデルと乗法型ホルト・ウィンタース季節モデルを当てはめる。各モデルは、容量計画・準備金計画のために95%信頼限界付きの12ヶ月分の点予測を生成する。季節ホルト・ウィンタースモデルは1～2ヶ月先で最も強い需要（約2,780～2,900件）を予測し、9ステップ目付近の底（約2,200件）まで緩んでいく一方、自己回帰ベースラインはより滑らかな減衰を予測する。両モデルとも信頼区間の幅は予測期間が延びるにつれて広がる。

## データソース

| データセット | 行数 | 粒度 | 主要変数 | 説明 |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | 暦月ごとに1行（2020年1月～2024年12月） | `date`（SAS日付、`MONYY7.`）、`claim_count`（数値） | 線形成長トレンド（月あたり約9件）、正弦波状の年次サイクル、加法的な12・1・2月の冬季悪天候急増、ガウスノイズ（`rand('normal')`）から構築した合成月次自動車保険請求件数。シード `20240531` により再現可能。 |

# 自動車保険月次請求件数の予測

個人向け損害保険会社の準備金設定・容量計画チームは、毎月報告される**自動車請求**の件数について今後の見通しを必要としている。この帳簿の請求件数はポートフォリオの拡大に伴って着実に増加し、氷・雪・日照時間の減少により衝突・ガラス破損の請求が増える毎冬に急増する。

このノートブックでは、完全な `PROC FORECAST` ワークフローを一通り実施する。

1. 現実的で完全に合成された月次請求件数系列を生成する。
2. トレンドと自己相関を捉える**ステップワイズ自己回帰（STEPAR）**ベースラインを当てはめる。
3. 12ヶ月の季節サイクルを明示的にモデル化する**乗法型ホルト・ウィンタース（WINTERS）**モデルを当てはめる。
4. 2つのモデルを比較し、先行き予測と信頼区間を解釈する。

すべてインラインの合成データ上で実行される -- 外部ファイルやネットワークアクセスは一切ない。

## ステップ1 -- 合成請求件数系列を生成する

以下の DATA ステップは**60ヶ月分の観測値**（2020年1月～2024年12月）を構築する。各月について、実際の自動車帳簿を反映する4つの要素を組み合わせる。

- **トレンド** -- エクスポージャーの増加に伴い月あたり約9件ずつ成長する、約1,800件のベースライン。
- **年次サイクル** -- なめらかな季節変動を与えるサイン／コサイン項。
- **冬季急増** -- 12月・1月・2月における加法的な増加。
- **ノイズ** -- 現実的な月々のばらつきを表す `rand('normal', 0, 90)`。

`call streaminit()` は乱数ストリームを固定し、ノートブックを再現可能にする。`date` 変数は `INTNX` で構築された真の SAS 日付であり `MONYY7.` でフォーマットされ、`ID date INTERVAL=MONTH` 文がそれを時間識別子として指定する。以下に印字される最初の14行は、おおよそ1,460件から2,450件の範囲に収まる。

In [1]:
データ claims;
    呼出 streaminit(20240531);
    繰返 i = 0 から 59;
        /* Build a real SAS month date so INTERVAL=MONTH aligns */
        date = intnx('month', '01JAN2020'd, i);
        書式 date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = Jan ... 12 = Dec */

        trend  = 1800 + 9 * i;               /* growing exposure base   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* ice/snow surge  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        もし claim_count < 0 なら claim_count = 0;
        出力;
    終了;
    保持 date claim_count;
実行;

処理 印刷 データ=claims(obs=14) 見出;
    見出 date = '月' claim_count = '請求件数';
    表題 '合成自動車請求件数の最初の14ヶ月';
実行;


                                                   合成自動車請求件数の最初の14ヶ月                                                    

  Obs      月          請求件数
    1  21915          2178
    2  21946          2281
    3  21975          2252
    4  22006          1974
    5  22036          2067
    6  22067          1805
    7  22097          1697
    8  22128          1619
    9  22159          1462
   10  22189          1688
   11  22220          1713
   12  22250          2008
   13  22281          2116
   14  22312          2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## ステップ2 -- ステップワイズ自己回帰ベースライン（METHOD=STEPAR）

`METHOD=STEPAR` はデフォルトである。まず時系列トレンドモデル（ここでは線形トレンドの `TREND=2`）を当てはめ、続いて残差に**ステップワイズ自己回帰**を適用し、有意性に基づいて AR ラグを追加・保持する。`AR=4` は候補となる自己回帰次数の上限を4ラグとする -- 短期の勢いを持つ月次系列には十分である。

ここで使われている主なオプション。

- `LEAD=12` -- データより12ヶ月先まで予測する。
- `ALPHA=0.05` -- 95%信頼限界。
- `OUTFULL` -- 過去実績（`ACTUAL`）行と予測期間の行を `OUT=` データセット内で積み重ねる（予測行のみではなく）。
- `OUTLIMIT` -- 信頼限界の下限／上限**列**（`L95_claim_count`、`U95_claim_count`）を追加する。
- `ID date INTERVAL=MONTH` -- 時間識別子を指定し、系列が月次であることを宣言する。

In [2]:
処理 forecast データ=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    変数 claim_count;
実行;

処理 印刷 データ=fc_stepar(obs=10) 見出;
    表題 'STEPAR出力：実績・当てはめ・予測行';
実行;


                                                   合成自動車請求件数の最初の14ヶ月                                                    

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                                  STEPAR出力：実績・当てはめ・予測行                                                  

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### `OUT=` データセットを読み解く

`OUT=` データセットは行を `_TYPE_` 列で積み重ね、信頼限界を側の**列**として持つ。

| 要素 | 種別 | 意味 |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | 行 | 60ヶ月の過去実績それぞれの観測 `claim_count`。 |
| `_TYPE_ = 'FORECAST'` | 行 | 予測期間の12件の点予測。 |
| `L95_claim_count` / `U95_claim_count` | 列 | 95%信頼限界の下限／上限。`FORECAST` 行にのみ値が入る（`ACTUAL` 行では欠測）。数値の水準は `ALPHA=` を反映する。 |

したがって、上に印字された `OUT=` は72行を保持する。60件の `ACTUAL` 履歴行に続く12件の `FORECAST` 予測期間行である。上に示した最初の10行はすべて `ACTUAL` であり、限界は予測行にのみ付属するため信頼限界列は欠測となっている。

> **エンジンに関する注記。** SAS の `OUTFULL` は各過去月について標本内の1期先 `FORECAST` 行も書き出し、`OUTRESID` は `_TYPE_='RESIDUAL'` 行を追加する。Jenner の PROC FORECAST はまだそれらの標本内当てはめ値／残差行を出力しない（ギャップテスト `400979` として追跡中）ため、このノートブックでは `ACTUAL` 履歴と先行き `FORECAST` 期間のみを読む。

## ステップ3 -- 季節ホルト・ウィンタースモデル（METHOD=WINTERS）

STEPARモデルはトレンドと短期の自己相関を捉えるが、繰り返し発生する12～2月の増加を明示的にはモデル化しない。明確な年次リズムを持つ系列には、**乗法型ホルト・ウィンタース**の方が適したツールである。

`METHOD=WINTERS` は系列を水準、線形トレンド（`TREND=2`）、**乗法的な季節係数**へと分解する。`SEASONS=12` は12期（月次）の季節サイクルを宣言する。ここでも95%限界付きの12ヶ月 `LEAD` と `OUTFULL OUTLIMIT` を要求し、出力を STEPAR の実行結果と揃える。

エンジンは（予測期間について `INTERVAL=MONTH` を尊重するのではなく -- ギャップテスト `400979`）予測 `ID` を1ステップごとに1単位ずつ進めるため、以下のセルでは印字された暦日に頼るのではなく、**先行月数（ステップ1～12）** で予測を確認する。

In [3]:
処理 forecast データ=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    変数 claim_count;
実行;

/* Keep the 12-month forward horizon and index it by step (1..12). */
データ horizon;
    設定 fc_winters;
    保存 months_ahead 0;
    もし _type_ = 'FORECAST';
    months_ahead + 1;
    保持 months_ahead claim_count l95_claim_count u95_claim_count;
実行;

処理 印刷 データ=horizon 見出;
    見出 months_ahead     = '経過月数'
          claim_count       = '予測請求件数'
          l95_claim_count   = '下限95%'
          u95_claim_count   = '上限95%';
    表題 'ホルト・ウィンタース12ヶ月先行き予測（ステップ別）';
実行;


                                                  STEPAR出力：実績・当てはめ・予測行                                                  

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                               ホルト・ウィンタース12ヶ月先行き予測（ステップ別）                                               

  Obs          経過月数              予測請求件数        下限95%        上限95%
    1             1          2783.07951  2577.844742  2988.314278
    2             2         2897.355557  2607.109764  3187.601349
    3             3         2805.272075  2449.795029   3160.74912
    4             4         2664.498049  2254.028514  3074.967585
    5             5         2628.810136  2169.891244  3087.729029
    6             6          2548.39319  2045.672732  3051.113649
    7             7         2391.649756    1848.6496  2934.649912
    8             8         2239.948352  1659.456768  2820.439936
    9             9         2197.109273  1581.404969 


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## ステップ4 -- 2つのモデルを直接比較する

季節モデルがその価値を発揮しているかどうかを判断する最も明確な方法は、その12ヶ月予測を STEPAR ベースラインとステップごとに並べることである。両方の `OUT=` データセットから `FORECAST` 行を取り出し、それぞれを先行月数でインデックス化し、結合して乖離が一目でわかるようにする。

2つの手法は水準についてはおおむね一致するが、**形状**については異なる。ホルト・ウィンタースは年次リズムを予測期間に持ち込み（初期の水準が高く、中間ホライズンで底を打ち、再び上昇する）、一方 STEPAR は -- 季節性を AR ラグを通じて間接的にのみモデル化するため -- より滑らかに系列平均へと減衰する。各ステップにおける両者のギャップこそ、STEPAR が見落としている季節シグナルである。

> SAS では通常、この妥当性チェックは `OUTRESID` の1期先残差（`_TYPE_='RESIDUAL'`）で行う。Jenner はまだそれらの行を出力しない（ギャップテスト `400979`）ため、代わりに2つの先行き予測を直接比較する -- エンジンが実際に出力するデータのみを使う診断である。

In [4]:
/* STEPAR horizon, indexed by months-ahead. */
データ stepar_h;
    設定 fc_stepar;
    保存 months_ahead 0;
    もし _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    保持 months_ahead stepar;
実行;

/* WINTERS horizon, indexed by months-ahead. */
データ winters_h;
    設定 fc_winters;
    保存 months_ahead 0;
    もし _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    保持 months_ahead winters;
実行;

/* Merge the two and show the seasonal gap STEPAR misses. */
データ 比較;
    結合 stepar_h winters_h;
    基準 months_ahead;
    seasonal_gap = winters - stepar;
実行;

処理 印刷 データ=比較 見出;
    見出 months_ahead = '経過月数'
          stepar        = 'STEPAR予測'
          winters       = 'Winters予測'
          seasonal_gap  = 'Winters - STEPAR';
    書式 stepar winters seasonal_gap 8.0;
    表題 'STEPAR対ホルト・ウィンタース：12ヶ月予測比較';
実行;


                                               STEPAR対ホルト・ウィンタース：12ヶ月予測比較                                               

  Obs          経過月数      STEPAR予測      Winters予測  Winters - STEPAR
    1             1          2619           2783               164
    2             2          2537           2897               361
    3             3          2384           2805               421
    4             4          2214           2664               450
    5             5          2089           2629               540
    6             6          2010           2548               539
    7             7          1982           2392               410
    8             8          1995           2240               245
    9             9          2031           2197               166
   10            10          2075           2354               280
   11            11          2115           2423               308
   12            12          2143           2798               655




NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## ステップ5 -- 全事業ラインを一度に予測する（BY処理）

実際の準備金設定業務では複数の商品を扱う。`line_of_business` でソートされたデータに対し、`BY` 文を使うことで `PROC FORECAST` は1回の呼び出しで**グループごとに独立した季節モデル**を当てはめる。ここでは自動車帳簿（基準となる件数が多い）と住宅帳簿（件数が少ない）を合成し、両方を6ヶ月先まで予測する。`OUTALL` はすべてのグループについて `ACTUAL` 履歴と `FORECAST` 期間の行一式を限界列とともに書き出し、以下の表では `FORECAST` 行のみを残す。

In [5]:
データ multi_book;
    呼出 streaminit(20240531);
    長さ line_of_business $12;
    繰返 lob = 1 から 2;
        もし lob = 1 なら line_of_business = '自動車';
        他            line_of_business = '住宅';
        繰返 i = 0 から 47;
            date = intnx('month', '01JAN2021'd, i);
            書式 date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            出力;
        終了;
    終了;
    保持 line_of_business date claim_count;
実行;

処理 並替 データ=multi_book;
    基準 line_of_business date;
実行;

処理 forecast データ=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    基準 line_of_business;
    id date interval=month;
    変数 claim_count;
実行;

処理 印刷 データ=fc_by(obs=12) 見出;
    条件 _type_ = 'FORECAST';
    見出 line_of_business = '事業ライン';
    表題 '事業ライン別6ヶ月予測';
実行;


                                               STEPAR対ホルト・ウィンタース：12ヶ月予測比較                                               


BY Group: line_of_business=住宅

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=自動車

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                                      事業ライン別6ヶ月予測                                                       

  Obs            事業ライン   DATE    _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  住宅               23742  FORECAST  1867.848547      1732.058284       2003.63881                     .
    2  住宅               23773  FORECAST  2027.978207      1835.941775      2220.014639                     .
    3  住宅               23801  FORECAST  1927.184503      1691.988868      2162.380137                     .
    4  住宅               23832  FORECAST  


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=住宅
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=自動車
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## 結果の解釈

**モデルが準備金設定チームに伝えること：**

- **STEPAR** は上昇トレンドと短期の勢いを再現するが、その12ヶ月予測はステップ1の約2,620件から中間ホライズンでおよそ1,980件まで滑らかに減衰し、その後約2,140件へと戻る -- 季節性を自己回帰ラグのみを通じて扱うため、鋭い冬季ピークを再現しない。手早いベースラインとしては妥当である。
- `SEASONS=12` を伴う**WINTERS** は年次リズムを乗法的な季節係数によって直接持ち込む。その予測は初期ホライズンで最も高く（ステップ1で約2,780件、ステップ2で約2,900件）、ステップ9付近の底（約2,200件）まで緩み、ステップ12までに再び約2,800件まで上昇する。STEPAR ベースラインに対しては、ホライズンの大半にわたって**150～660件高い**（ステップ4の `Winters - STEPAR` 列を参照）-- そのギャップこそ自己回帰モデルが見落としている季節シグナルである。
- **95%信頼区間**（`L95_*`／`U95_*` 列、`ALPHA=` により制御）はホライズンが延びるにつれて広がる -- WINTERS モデルではステップ1の幅約410件からステップ12では約1,420件まで -- これは、ホライズン後半の推定値が近い将来の推定値よりも大きな不確実性を伴うという正直なシグナルである。準備金担当者は点予測だけでなく上限に対して資本を保持すべきである。
- **BY処理**により、自動車帳簿と住宅帳簿の予測が1回のパスでそれぞれ独自の季節フィットとともに生成される。自動車帳簿はおよそ2,510～2,600件のレンジで予測され、住宅帳簿は1,870～2,030件付近に位置し、各ラインは自らの水準と季節形状を保持する -- このパターンはポートフォリオ内のすべての商品に拡張できる。

**結論：** 明確な年次サイクルを持つ請求件数系列には `METHOD=WINTERS SEASONS=12` が定石の選択であり、STEPAR ベースラインは有用な健全性チェックであり、`OUTFULL`／`OUTLIMIT` とステップごとのモデル比較により、アクチュアリーは不確実性帯とともに将来の準備金を見積もることができる。

> **エンジン忠実度に関する注記。** このノートブックは現行の Jenner PROC FORECAST の2つの制限（ギャップテスト `400979`）を記録している。予測期間の `ID` が `INTERVAL=MONTH` ではなくステップごとに1単位ずつ進むため、印字される予測日付は意図した2025年の暦月ではない（そのため、ホライズンはステップ単位で確認する）。また `OUTRESID`／`OUTALL` はまだ1期先の `_TYPE_='RESIDUAL'` 行を出力しないため、残差診断は2モデルの直接比較に置き換えている。点予測と信頼限界そのものは実際に生成されており、上記の説明の根拠となっている。